# CloudGuardian — prioritization model

Reproduces the Week 2 prioritization pipeline: loads Prowler CSPM findings,
scores each FAIL finding on `severity x exposure x blast_radius`, and writes
the ranked results into `consolidated_findings.db` (SQLite).

**TRACKED MISCONFIGURATIONS:** This notebook filters to only the 10 tracked
misconfigurations (MC-01 through MC-10). Any additional findings from Prowler
(e.g., root account MFA) are excluded from the prioritization and database.

Replace `data/sample_findings.csv` with your real Prowler export
(192 findings) to reproduce the full run — the scoring logic is identical,
this notebook just wraps `build_db.py` with inline visibility into each step.


In [1]:
import csv
import sqlite3
from pathlib import Path
from collections import Counter

import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from build_db import (
    SEVERITY_WEIGHTS, EXPOSURE_RULES, BLAST_RADIUS_BY_RESOURCE_TYPE,
    score_exposure, score_blast_radius, load_and_score, write_db,
)

CSV_PATH = PROJECT_ROOT / "data/sample_findings.csv"
DB_PATH = PROJECT_ROOT / "db/consolidated_findings.db"
SCHEMA_PATH = PROJECT_ROOT / "db/schema.sql"

# --- TRACKED MISCONFIGURATIONS ---
TRACKED_MCS = ['MC-01', 'MC-02', 'MC-03', 'MC-04', 'MC-05', 
               'MC-06', 'MC-07', 'MC-08', 'MC-09', 'MC-10']


## 1. Load raw findings

In [2]:
with CSV_PATH.open() as f:
    raw_rows = list(csv.DictReader(f))

print(f"{len(raw_rows)} total findings loaded from {CSV_PATH.name}")
print(Counter(r["status"] for r in raw_rows))
print(Counter(r["severity"] for r in raw_rows))

# --- FILTER: Keep only our 10 tracked misconfigurations ---
filtered_rows = [r for r in raw_rows if r.get('misconfig_id') in TRACKED_MCS]
print(f"\n✅ Filtered to {len(filtered_rows)} tracked misconfigurations")
print(f"Tracked MCs found: {Counter(r.get('misconfig_id') for r in filtered_rows)}")

# Replace raw_rows with filtered version for the rest of the notebook
raw_rows = filtered_rows


## 2. Scoring model

`priority_score = severity_score x exposure_score x blast_radius`

- **severity_score** — Prowler's own severity, weighted 1-10
- **exposure_score** — keyword-matched against the check_id (public access,
  open ingress, missing MFA, etc. score highest)
- **blast_radius** — fixed per resource type (IAM/RDS score highest — a
  compromised credential or database has the widest downstream impact)


In [3]:
print("Severity weights:", SEVERITY_WEIGHTS)
print()
print("Exposure rules (keyword -> score):")
for kw, score in EXPOSURE_RULES:
    print(f"  {kw:<35} {score}")
print()
print("Blast radius by resource type:", BLAST_RADIUS_BY_RESOURCE_TYPE)


## 3. Score and rank (filtered to tracked MCs)

In [4]:
# Load and score all findings, then filter to tracked MCs
all_scored = load_and_score(CSV_PATH)
scored_rows = [r for r in all_scored if r.get('misconfig_id') in TRACKED_MCS]
fails = [r for r in scored_rows if r["status"].upper() == "FAIL"]

print(f"{len(fails)} FAIL findings ranked by priority\n")
print(f"{'Rank':<5}{'Score':<8}{'Sev':<10}{'MC':<7}{'Check':<45}")
for r in fails:
    print(f"{r['priority_rank']:<5}{r['priority_score']:<8.0f}{r['severity']:<10}"
          f"{(r['misconfig_id'] or '-'): <7}{r['check_id'][:44]:<45}")


## 4. Persist to the consolidated findings database

In [5]:
write_db(scored_rows, DB_PATH, SCHEMA_PATH)
print(f"Wrote {len(scored_rows)} findings to {DB_PATH}")


## 5. Verify — query back from SQLite

In [6]:
conn = sqlite3.connect(DB_PATH)
top = conn.execute('''
    SELECT priority_rank, severity, misconfig_id, priority_score, title
    FROM findings
    WHERE status = "FAIL"
    ORDER BY priority_rank
    LIMIT 10
''').fetchall()

for row in top:
    print(row)

conn.close()


## 6. Next step: remediation

For each top-ranked finding with a `misconfig_id`, the corresponding prompt
in `prompts/prompt_library.md` generates the remediation guidance, which
feeds either an auto-remediation Lambda (guardrailed, reversible changes)
or the human-approval queue (higher-risk changes like IAM/network access).
See `../lambdas/` for the remediation functions.
